In [1]:
import numpy as np

weights = np.array([
    [-1.8, -0.9, 0.0, 0.7, 1.5],
    [-2.4, -0.3, 0.2, 1.1, 2.0]
], dtype=np.float32)

activations = np.array([
    [0.0, 0.3, 0.8, 1.4, 2.1],
    [0.1, 0.6, 1.0, 1.8, 3.2]
], dtype=np.float32)

In [ ]:
def quantize_tensor(tensor, scale, zero_point):
    q=np.round(tensor/scale) + zero_point
    q=np.clip(q,-128,127)
    return q.astype(np.int8)

In [3]:
def symmetric_quantize(tensor):
    x_min = np.min(tensor)
    x_max = np.max(tensor)
    scale = max(abs(x_min), abs(x_max)) / 127
    zero_point = 0
    quantized_tensor = np.round(tensor / scale)
    quantized_tensor = np.clip(quantized_tensor, -127, 127)
    quantized_tensor = quantized_tensor.astype(np.int8)
    return quantized_tensor, scale, zero_point

In [4]:
q_weights, scale, zp = symmetric_quantize(weights)

print("Scale:", scale)
print("Zero Point:", zp)
print("Quantized Weights:")
print(q_weights)

Scale: 0.018897638
Zero Point: 0
Quantized Weights:
[[ -95  -48    0   37   79]
 [-127  -16   11   58  106]]


In [5]:
def asymmetric_quantize(tensor):
    x_min = np.min(tensor)
    x_max = np.max(tensor)
    if x_max == x_min:
        scale = 1.0
        zero_point = 0
        quantized_tensor = np.zeros_like(tensor, dtype=np.int8)
        return quantized_tensor, scale, zero_point
    scale = (x_max - x_min) / 255
    zero_point = round(-128 - (x_min / scale))
    zero_point = np.clip(zero_point, -128, 127)

    quantized_tensor = np.round(tensor / scale) + zero_point
    quantized_tensor = np.clip(quantized_tensor, -128, 127)
    quantized_tensor = quantized_tensor.astype(np.int8)

    return quantized_tensor, scale, zero_point

In [6]:
q_weights_asym, scale_asym, zp_asym = asymmetric_quantize(weights)

print("Scale:", scale_asym)
print("Zero Point:", zp_asym)
print("Quantized Weights:")
print(q_weights_asym)

Scale: 0.017254902
Zero Point: 11
Quantized Weights:
[[ -93  -41   11   52   98]
 [-128   -6   23   75  127]]


In [7]:
def dequantize(quantized_tensor, scale, zero_point):
    return (quantized_tensor - zero_point) * scale


deq_weights_sym = dequantize(q_weights, scale, zp)

print("Dequantized Weights:")
print(deq_weights_sym)

Dequantized Weights:
[[-1.7952756  -0.9070866   0.          0.6992126   1.4929134 ]
 [-2.4        -0.3023622   0.20787401  1.096063    2.0031495 ]]


In [8]:
def calculate_metrics(original, dequantized, quantized):
    error = original - dequantized
    mae = np.mean(np.abs(error))
    mse = np.mean(error ** 2)
    max_error = np.max(np.abs(error))

    sat_min = np.sum(quantized == -128)
    sat_max = np.sum(quantized == 127)
    sat_total = sat_min + sat_max

    return mae, mse, max_error, sat_min, sat_max, sat_total

In [9]:
q_w_sym, scale_w_sym, zp_w_sym = symmetric_quantize(weights)

deq_w_sym = dequantize(q_w_sym, scale_w_sym, zp_w_sym)

mae_w_sym, mse_w_sym, max_w_sym, satmin_w_sym, satmax_w_sym, sattotal_w_sym = calculate_metrics(
    weights,
    deq_w_sym,
    q_w_sym
)

In [10]:
q_w_asym, scale_w_asym, zp_w_asym = asymmetric_quantize(weights)

deq_w_asym = dequantize(q_w_asym, scale_w_asym, zp_w_asym)

mae_w_asym, mse_w_asym, max_w_asym, satmin_w_asym, satmax_w_asym, sattotal_w_asym = calculate_metrics(
    weights,
    deq_w_asym,
    q_w_asym
)

In [11]:
q_a_sym, scale_a_sym, zp_a_sym = symmetric_quantize(activations)

deq_a_sym = dequantize(q_a_sym, scale_a_sym, zp_a_sym)

mae_a_sym, mse_a_sym, max_a_sym, satmin_a_sym, satmax_a_sym, sattotal_a_sym = calculate_metrics(
    activations,
    deq_a_sym,
    q_a_sym
)

In [12]:
q_a_asym, scale_a_asym, zp_a_asym = asymmetric_quantize(activations)

deq_a_asym = dequantize(q_a_asym, scale_a_asym, zp_a_asym)

mae_a_asym, mse_a_asym, max_a_asym, satmin_a_asym, satmax_a_asym, sattotal_a_asym = calculate_metrics(
    activations,
    deq_a_asym,
    q_a_asym
)

In [13]:
print("\n===== Weights - Symmetric =====")
print("Scale:", scale_w_sym)
print("Zero Point:", zp_w_sym)
print("MAE:", mae_w_sym)
print("MSE:", mse_w_sym)
print("Maximum Error:", max_w_sym)
print("Saturated at -128:", satmin_w_sym)
print("Saturated at 127:", satmax_w_sym)
print("Total Saturation:", sattotal_w_sym)

print("\n===== Weights - Asymmetric =====")
print("Scale:", scale_w_asym)
print("Zero Point:", zp_w_asym)
print("MAE:", mae_w_asym)
print("MSE:", mse_w_asym)
print("Maximum Error:", max_w_asym)
print("Saturated at -128:", satmin_w_asym)
print("Saturated at 127:", satmax_w_asym)
print("Total Saturation:", sattotal_w_asym)

print("\n===== Activations - Symmetric =====")
print("Scale:", scale_a_sym)
print("Zero Point:", zp_a_sym)
print("MAE:", mae_a_sym)
print("MSE:", mse_a_sym)
print("Maximum Error:", max_a_sym)
print("Saturated at -128:", satmin_a_sym)
print("Saturated at 127:", satmax_a_sym)
print("Total Saturation:", sattotal_a_sym)

print("\n===== Activations - Asymmetric =====")
print("Scale:", scale_a_asym)
print("Zero Point:", zp_a_asym)
print("MAE:", mae_a_asym)
print("MSE:", mse_a_asym)
print("Maximum Error:", max_a_asym)
print("Saturated at -128:", satmin_a_asym)
print("Saturated at 127:", satmax_a_asym)
print("Total Saturation:", sattotal_a_asym)


===== Weights - Symmetric =====
Scale: 0.018897638
Zero Point: 0
MAE: 0.0037007749
MSE: 2.1638e-05
Maximum Error: 0.007874012
Saturated at -128: 0
Saturated at 127: 0
Total Saturation: 0

===== Weights - Asymmetric =====
Scale: 0.017254902
Zero Point: 11
MAE: 0.003803923726081848
MSE: 2.1237957060186995e-05
Maximum Error: 0.007450995966792107
Saturated at -128: 1
Saturated at 127: 1
Total Saturation: 2

===== Activations - Symmetric =====
Scale: 0.02519685
Zero Point: 0
MAE: 0.0052755615
MSE: 4.4825596e-05
Maximum Error: 0.011023641
Saturated at -128: 0
Saturated at 127: 1
Total Saturation: 1

===== Activations - Asymmetric =====
Scale: 0.012549019
Zero Point: -128
MAE: 0.0026274432428181173
MSE: 1.1118679153918326e-05
Maximum Error: 0.005490198731422424
Saturated at -128: 1
Saturated at 127: 1
Total Saturation: 2


In [14]:
print("\n===== WEIGHTS =====")
print("Original:")
print(weights)

print("\nSymmetric Quantized:")
print(q_w_sym)

print("\nSymmetric Dequantized:")
print(deq_w_sym)

print("\nAsymmetric Quantized:")
print(q_w_asym)

print("\nAsymmetric Dequantized:")
print(deq_w_asym)


print("\n\n===== ACTIVATIONS =====")
print("Original:")
print(activations)

print("\nSymmetric Quantized:")
print(q_a_sym)

print("\nSymmetric Dequantized:")
print(deq_a_sym)

print("\nAsymmetric Quantized:")
print(q_a_asym)

print("\nAsymmetric Dequantized:")
print(deq_a_asym)


===== WEIGHTS =====
Original:
[[-1.8 -0.9  0.   0.7  1.5]
 [-2.4 -0.3  0.2  1.1  2. ]]

Symmetric Quantized:
[[ -95  -48    0   37   79]
 [-127  -16   11   58  106]]

Symmetric Dequantized:
[[-1.7952756  -0.9070866   0.          0.6992126   1.4929134 ]
 [-2.4        -0.3023622   0.20787401  1.096063    2.0031495 ]]

Asymmetric Quantized:
[[ -93  -41   11   52   98]
 [-128   -6   23   75  127]]

Asymmetric Dequantized:
[[-1.79450981 -0.89725491  0.          0.70745098  1.50117648]
 [-2.39843138 -0.29333333  0.20705882  1.10431373  2.00156864]]


===== ACTIVATIONS =====
Original:
[[0.  0.3 0.8 1.4 2.1]
 [0.1 0.6 1.  1.8 3.2]]

Symmetric Quantized:
[[  0  12  32  56  83]
 [  4  24  40  71 127]]

Symmetric Dequantized:
[[0.        0.3023622 0.8062992 1.4110236 2.0913386]
 [0.1007874 0.6047244 1.007874  1.7889764 3.2      ]]

Asymmetric Quantized:
[[-128 -104  -64  -16   39]
 [-120  -80  -48   15  127]]

Asymmetric Dequantized:
[[0.         0.30117647 0.80313724 1.40549017 2.09568624]
 [0.

In [15]:
outlier_tensor = np.array(
    [-0.5, -0.2, 0.0, 0.3, 0.7, 12.0],
    dtype=np.float32
)

q_sym, scale_sym, zp_sym = symmetric_quantize(outlier_tensor)

deq_sym = dequantize(q_sym, scale_sym, zp_sym)

mae_sym, mse_sym, max_sym, satmin_sym, satmax_sym, sattotal_sym = calculate_metrics(
    outlier_tensor,
    deq_sym,
    q_sym
)

q_asym, scale_asym, zp_asym = asymmetric_quantize(outlier_tensor)

deq_asym = dequantize(q_asym, scale_asym, zp_asym)

mae_asym, mse_asym, max_asym, satmin_asym, satmax_asym, sattotal_asym = calculate_metrics(
    outlier_tensor,
    deq_asym,
    q_asym
)

In [16]:
without_outlier = np.array(
    [-0.5, -0.2, 0.0, 0.3, 0.7],
    dtype=np.float32
)
q_sym2, scale_sym2, zp_sym2 = symmetric_quantize(without_outlier)

deq_sym2 = dequantize(q_sym2, scale_sym2, zp_sym2)

mae_sym2, mse_sym2, max_sym2, satmin_sym2, satmax_sym2, sattotal_sym2 = calculate_metrics(
    without_outlier,
    deq_sym2,
    q_sym2
)
q_asym2, scale_asym2, zp_asym2 = asymmetric_quantize(without_outlier)

deq_asym2 = dequantize(q_asym2, scale_asym2, zp_asym2)

mae_asym2, mse_asym2, max_asym2, satmin_asym2, satmax_asym2, sattotal_asym2 = calculate_metrics(
    without_outlier,
    deq_asym2,
    q_asym2
)

In [17]:
print("\n===== WITH OUTLIER - SYMMETRIC =====")
print("Scale:", scale_sym)
print("Zero Point:", zp_sym)
print("MAE:", mae_sym)
print("MSE:", mse_sym)
print("Maximum Error:", max_sym)

print("\n===== WITH OUTLIER - ASYMMETRIC =====")
print("Scale:", scale_asym)
print("Zero Point:", zp_asym)
print("MAE:", mae_asym)
print("MSE:", mse_asym)
print("Maximum Error:", max_asym)

print("\n===== WITHOUT OUTLIER - SYMMETRIC =====")
print("Scale:", scale_sym2)
print("Zero Point:", zp_sym2)
print("MAE:", mae_sym2)
print("MSE:", mse_sym2)
print("Maximum Error:", max_sym2)

print("\n===== WITHOUT OUTLIER - ASYMMETRIC =====")
print("Scale:", scale_asym2)
print("Zero Point:", zp_asym2)
print("MAE:", mae_asym2)
print("MSE:", mse_asym2)
print("Maximum Error:", max_asym2)


===== WITH OUTLIER - SYMMETRIC =====
Scale: 0.09448819
Zero Point: 0
MAE: 0.015616802
MSE: 0.000440511
Maximum Error: 0.038582683

===== WITH OUTLIER - ASYMMETRIC =====
Scale: 0.04901961
Zero Point: -118
MAE: 0.007189571236570676
MSE: 7.176779367817215e-05
Maximum Error: 0.013725467026233673

===== WITHOUT OUTLIER - SYMMETRIC =====
Scale: 0.005511811
Zero Point: 0
MAE: 0.0011023671
MSE: 2.1080245e-06
Maximum Error: 0.0023622215

===== WITHOUT OUTLIER - ASYMMETRIC =====
Scale: 0.0047058826
Zero Point: -22
MAE: 0.0011764748021960258
MSE: 1.9377220808665733e-06
Maximum Error: 0.0023529324680566788


In [18]:
print("\nPer-element Error (Symmetric):")
print(np.abs(outlier_tensor - deq_sym))

print("\nPer-element Error (Asymmetric):")
print(np.abs(outlier_tensor - deq_asym))


Per-element Error (Symmetric):
[0.02755904 0.01102363 0.         0.01653546 0.03858268 0.        ]

Per-element Error (Asymmetric):
[0.00980391 0.00392157 0.         0.00588236 0.01372547 0.00980412]
